<style>
div.output_scroll {
    height: auto !important;
    max-height: none !important;
}
</style>

# Analyse et Nettoyage du dataset UE

Seuls les véhicules de l'année 2023 et de la France ont été exportés à partir de https://www.eea.europa.eu/data-and-maps/data/co2-cars-emission-20

# <font color='#3585CD'>Importation des librairies</font>

In [ ]:
import warnings
warnings.filterwarnings('ignore')
warnings.warn('DelftStack')
warnings.warn('Do not show this message')

import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

import plotly.figure_factory as ff
import plotly.express as px
import plotly.graph_objects as go
from google.colab import drive

# <font color='#3585CD'>Chargement des données</font>

## Chargement du dataset principal

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

dataset_path = "/content/drive/MyDrive/data_2023_FR.csv"
df = pd.read_csv(dataset_path)


In [ ]:
# Suppression des espaces accidentels dans les noms des colonnes comme "Fuel consumption "
df.columns = df.columns.str.strip()

# <font color='#3585CD'>Premières analyses</font>

## Informations sur le dataset

In [ ]:
print("\nAperçu du dataset :")
print(df.info())

## Satistiques descriptives

In [ ]:
print("\nStatistiques descriptives :")
display(df.describe(include='all').T)

## Nombre de valeurs uniques par colonne

In [ ]:
# Calculer le nombre de valeurs uniques par colonne et renommer la colonne
df_unique_values = df.nunique().sort_values(ascending=False).reset_index()
df_unique_values.columns = ["colonne", "nombre de valeurs uniques"]

df_unique_values

## S'assurer des types d'entrée dans chaque colonne

In [ ]:
# Vérification du dtype de chaque colonne et de la validité des données
for column in df.columns:
    print(f"Colonne: {column}")
    print(f"Type actuel de la colonne : {df[column].dtype}")

    # Si le type est 'object', vérifier si ce sont des chaînes de caractères qui devraient être numériques
    if df[column].dtype == 'object':
        # Tentative de conversion des valeurs de la colonne en numériques
        try:
            # Convertir la colonne en numérique, les erreurs seront ignorées
            converted = pd.to_numeric(df[column], errors='coerce')

            # Si des valeurs sont converties en NaN, cela signifie que certaines valeurs étaient des chaînes non numériques
            if converted.isnull().sum() > 0:
                print(f" - Il y a {converted.isnull().sum()} valeurs non numériques dans cette colonne.")
                print(f" - Quelques valeurs non numériques : {df[column].head()}")
            else:
                print(f" - Toutes les valeurs sont numériques (ou peuvent être converties).")

            # Afficher quelques exemples de valeurs converties
            print(f" - Exemple de valeurs converties : {converted.head()}")
        except Exception as e:
            print(f" - Erreur lors de la tentative de conversion : {e}")
    print("\n")

## Mettre la colonne <strong>ID</strong> en index !

In [ ]:
# mettre le colonne ID en index !
df.set_index('ID', inplace=True)


In [ ]:
df.head(20)

## Calculer le nombre de valeurs uniques par colonne apres l'indexation de <strong> ID </strong>

In [ ]:
# Calculer le nombre de valeurs uniques par colonne et renommer la colonne
df_unique_values = df.nunique().sort_values(ascending=False).reset_index()
df_unique_values.columns = ["colonne", "nombre de valeurs uniques"]

df_unique_values

## conversion de la colonne <strong> Date of registration </strong> en datetime

In [ ]:
# Convertir la colonne 'Date of registration' en datetime
df['Date of registration'] = pd.to_datetime(df['Date of registration'], errors='coerce')

## Créer une nouvelle colonne **Year** (Dans le cas où nous ajoutons des données)

In [ ]:
# Extraire l'année et créer une nouvelle colonne
df['Year of registration'] = df['Date of registration'].dt.year

## Analyse des valeurs manquantes

In [ ]:
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]  # Filtrer uniquement les colonnes avec des valeurs manquantes
missing_values_percentage = (missing_values / len(df)) * 100

data_na = pd.DataFrame({'Valeurs manquantes': missing_values, 'Pourcentage': missing_values_percentage})
print(data_na)

# Gestion des doublons

In [ ]:
df.duplicated().sum()

# Afficher un aperçu des colonnes en double

In [ ]:
duplicated =  df.duplicated(keep=False)
some_duplicates = df[df.duplicated()].sort_values(by=df.columns.to_list()).head(20)
print(f"Le DataFrame contient une ou plusieurs lignes dupliquées, par exemple :\n{some_duplicates}")

## Nombre de doublon par colonne

In [ ]:
# Afficher le nombre de doublons par colonne
for column in df.columns:
    duplicates_count = df[column].duplicated().sum()  # Compter le nombre de doublons pour chaque colonne
    if duplicates_count > 0:
        print(f"La colonne '{column}' a  : {duplicates_count}")
    else:
        print(f"La colonne '{column}' n'a pas de doublons.")

## suppression des doublons

In [ ]:
# sppression des doublons en actualisant l'index
df = df.drop_duplicates().reset_index(drop=True)



In [ ]:
print(df.duplicated().sum())


In [ ]:
print(df.columns)

In [ ]:
print(data_na)

#Suppression de colonnes

Nous pouvons dès à présent ces colonnes qui ont un taux de valeurs manquantes supérieur à 70% :

*   At2 (mm)
*   W (mm)
*   MMS
*   Vf
*   De
*   Ernedc (g/km)
*   At1 (mm)
*   Enedc (g/km)
*   RLFI
*   z (Wh/km)
*   Electric range (km)

In [ ]:
df = df.drop(columns=['At2 (mm)',	'W (mm)',	'MMS',	'Vf',	'De', 'Ernedc (g/km)',	'At1 (mm)',	'Enedc (g/km)',	'RLFI',	'z (Wh/km)', 'Electric range (km)'], axis=1)

# <font color='#3585CD'>Suppresion des colonnes non pertinentes</font>

Certaines colonnes n'ont pas d'intérêt à être gardées :

*   IT <strong> * (gardé pour l'instant)</strong>
*   Erwltp (g/km) (dépreciée) <strong>* (gardé pour l'instant)</strong>
*   ID : identifiant du véhicule (indexée)
*   Country : notre dataset est une extraction des véhicules de France
*   VFN : n'a pas de norme universelle et comporte trop de valeurs
*   Tan : trop de valeurs et sans intérêt pour notre projet
*   T : trop de valeurs et sans intérêt pour notre projet
*   Va : trop de valeurs et sans intérêt pour notre projet
*   Ve : trop de valeurs et sans intérêt pour notre projet
*   Status : n'a qu'une seule valeur et ne varie pas
*   Year : 1 seule valeur
*   Date of registration : sans intérêt pour notre projet     <strong> *(gardé pour l'instant)</strong>
*   Fm : redondant avec Ft
*   Cr : nous avons 2 catégories (M1, M1G). M1G est une sous-catégorie de M1 réservée aux véhicules tout-terrain. Nous pouvons conclure que tous les véhicules sont de catégorie M1
*   Ct : idem que Cr
*   ech : sans intérêt pour notre projet
*   Mp : redondant, se retrouve dans une autre colonne
*   Man : redondant avec Mk
*   r : n'a qu'une seule valeur
*   Mh : redondant avec Mk


In [ ]:
df = df.drop(columns=['Country', 'VFN', 'Tan', 'T', 'Va', 'Ve', 'Status', 'year', 'Fm', 'Cr', 'Ct', 'ech', 'Mp', 'Man', 'r', 'Mh'], axis=1)

# <font color='#3585CD'>Traitement des valeurs manquantes</font>

 ## Affichage du pourcentage de valeurs manquantes par colonne

> Ajouter une citation



In [ ]:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_percentage = missing_percentage[missing_percentage > 0]  # Filtrer les colonnes avec NaN
print(missing_percentage)

In [ ]:
def display_missing_values(df):
    """
    Affiche le nombre et le pourcentage de valeurs manquantes pour chaque colonne du DataFrame.

    Args:
        df (pd.DataFrame): Le DataFrame à analyser.

    Returns:
        pd.DataFrame: Un DataFrame indiquant le nombre et le pourcentage de valeurs manquantes par colonne.
    """
    missing_values = df.isnull().sum()  # Nombre de valeurs manquantes par colonne
    missing_percentage = (missing_values / len(df)) * 100  # Pourcentage de valeurs manquantes

    # Création d'un DataFrame pour afficher les résultats
    missing_data = pd.DataFrame({
        "Valeurs manquantes": missing_values,
        "Pourcentage (%)": missing_percentage
    })

    # Trier par ordre décroissant de valeurs manquantes
    missing_data = missing_data[missing_data["Valeurs manquantes"] > 0].sort_values(by="Valeurs manquantes", ascending=False)

    # Affichage des résultats
    print("\n🔍 Aperçu des valeurs manquantes :")
    print(missing_data)

    return missing_data

In [ ]:
data_na = display_missing_values(df)
data_na

# Nombre de lignes restantes

In [ ]:
print(f"Nombre total de lignes : {df.shape[0]}")

On a 90 182 valeurs manquantes dans les colonnes Fuel consumption et Cylindrée moteur (cm³). On va commencer par supprimer les lignes manquantes de Fuel consumption et voir si l'équilibre du dataset s'améliore.

# Suppression des valeurs manquantes dans la colonne "Fuel consumption"

In [ ]:
df = df.dropna(subset=["Fuel consumption"])


In [ ]:
data_na = display_missing_values(df)
data_na

# Nombre de lignes restantes

In [ ]:
print(f"Nombre total de lignes : {df.shape[0]}")

In [ ]:
print(df.columns)

# Vérification de la corrélation entre la colonne IT (catégorielle) et les émissions de CO2

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Initialiser l'encodeur
encoder = LabelEncoder()

# Encoder la colonne IT
df['IT_encoded'] = encoder.fit_transform(df['IT'])

# Calculer la corrélation entre la colonne encodée IT et CO2
correlation_IT_CO2 = df[['IT_encoded', 'Erwltp (g/km)', 'Ewltp (g/km)']].corr()

print(correlation_IT_CO2)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Matrice de corrélation
corr_matrix = df[['IT_encoded', 'Erwltp (g/km)', 'Ewltp (g/km)']].corr()

# Affichage du heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matrice de Corrélation entre IT_encoded, Erwltp (g/km), Ewltp (g/km)')
plt.show()


<strong> IT_encoded et CO2:</strong> Une corrélation négative modérée (-0.52) suggère que les véhicules avec des technologies innovantes (IT) ont tendance à émettre moins de CO2.

<strong>impact_innovaion et CO2 :</strong> Avec une corrélation de 0,22, on observe une légère relation positive entre l'impact de l'innovation et les émissions de CO2. Cela suggère que les technologies innovantes pourraient avoir une certaine influence, mais leur effet reste relativement modéré.

In [ ]:
# Sélectionner uniquement les colonnes numériques
numeric_df = df.select_dtypes(include=['float64', 'int64'])

# Calculer la matrice de corrélation uniquement sur les colonnes numériques
correlation_matrix = numeric_df.corr()

# Afficher la matrice de corrélation
import seaborn as sns
import matplotlib.pyplot as plt

# Tracer la heatmap de la matrice de corrélation
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Matrice de Corrélation")
plt.show()


# <strong> Décision concernat les variables impact_innovation et type_d'innovation

Après vérification des corrélations, nous avons observé une corrélation modérée (0,22) entre les émissions de CO2 (CO2) et l’impact des innovations (impact_innovation). Bien que cette relation ne soit pas négligeable, elle reste relativement faible pour avoir un impact significatif sur notre modèle. Nous allons donc évaluer sa pertinence, mais elle pourrait être supprimée si elle n'apporte pas de valeur ajoutée à l’analyse.

En revanche, la colonne (type_d'innovation) présente une corrélation plus marquée de -0,52 avec les émissions de CO2. Nous allons donc la conserver et approfondir son analyse, car elle pourrait fournir des informations intéressantes sur l’impact des technologies innovantes dans la réduction des émissions.

## <strong> Gestion de valeurs manquantes restantes

On va supprimer les valeurs manquantes de la colonne "type_innovation" et vérifier si cela permet également d’éliminer les lignes contenant des valeurs manquantes dans la colonne "impact_innovation".

In [ ]:
df = df.dropna(subset=["type_innovation"])

In [ ]:
data_na = display_missing_values(df)
data_na

# Nombre de lignes restantes

In [ ]:
print(f"Nombre total de lignes : {df.shape[0]}")

In [ ]:
df.info()

# <font color='#3585CD'>Renommage des colonnes</font>

Pour plus de compréhension, nous allons renommer les colonnes :



*   Mk : Marque
*   Cn : Modele
*   Mt : Masse
*   Ewltp (g/km) : Co2
*   Ft : Carburant
*   ec (cm3) : Cylindree moteur
*   ep (KW) : Puissance moteur
*   Fuel consumption : Consommation carburant


In [ ]:
renommage = {
    'IT': 'type_innovation',
    'Erwltp (g/km)': 'impact_innovaion',
    'Mk': 'Marque',
    'Cn': 'Modèle',
    'm (kg)' : 'Masse à vide',
    'Mt': 'Masse totale',
    'Ewltp (g/km)': 'CO2',
    'Ft': 'Carburant',
    'ec (cm3)': 'Cylindrée moteur',
    'ep (KW)': 'Puissance moteur',
    'Fuel consumption': 'Consommation carburant',
    'Date of registration': 'Date_enregistrement',
    'Year of registration': 'Année'
}

# Application du renommage
df.rename(columns=renommage, inplace=True)

In [ ]:
df.info()

# Distribution des marques de voiture

In [ ]:
# Répartition des marques de voiture
brand_counts = df['Marque'].value_counts()

# Calcul du pourcentage de chaque marque
brand_percentage = (brand_counts / len(df)) * 100

# Affichage des résultats
print("Répartition des marques de voitures dans le dataset :")
print(brand_percentage)

# Visualisation sous forme de graphique à barres
plt.figure(figsize=(12, 6))
sns.barplot(x=brand_percentage.index, y=brand_percentage.values)
plt.xticks(rotation=90)
plt.xlabel("Marque")
plt.ylabel("Pourcentage")
plt.title("Répartition des marques de voitures")
plt.show()

# <font color='#3585CD'>Faut-il garder les véhicules électriques et hydrogènes ?</font>
Notre objectif est de prédire les émissions directes de CO2. Garder les véhicules électriques et hydrogènes risque de biaiser notre modèle. Nous allons donc exclure les véhicules électriques de notre dataset

In [ ]:
# Nous excluons les véhicules électriques et hydrogènes
df = df[(df["Carburant"] != "electric") & (df["Carburant"] != "hydrogen")]

In [ ]:
df.shape

In [ ]:
# Vérification des émissions de CO2 des véhicules hydrogènes
df_hydrogen = df[df["Carburant"] == "hydrogen"]
df_hydrogen['CO2'].value_counts()

In [ ]:
df_corr = df.copy()  # Copie du DataFrame pour ne pas modifier l'original

# Encodage des variables catégorielles avec la moyenne de CO2
df_corr["Marque_encoded"] = df_corr.groupby("Marque")["CO2"].transform("mean")
df_corr["Modèle_encoded"] = df_corr.groupby("Modèle")["CO2"].transform("mean")
df_corr["Carburant_encoded"] = df_corr.groupby("Carburant")["CO2"].transform("mean")

# Calculer la corrélation avec CO2
correlation_values = df_corr[["Marque_encoded", "Modèle_encoded", "Carburant_encoded", "CO2"]].corr()

# Afficher uniquement la corrélation avec CO2
print(correlation_values["CO2"])


# <font color='#3585CD'>Distribution des variables catégorielles</font>

## Analyse par marque

### Valeurs uniques

In [ ]:
sorted(df['Marque'].unique())

In [ ]:
# Afficher toutes les valeurs uniques et leurs occurrences
pd.set_option('display.max_rows', None)  # Permet d'afficher toutes les lignes
print(df['Marque'].value_counts().sort_index())  # Trie par ordre alphabétique


### Remplacement

Certaines valeurs peuvent être regroupées :


*   'MERCEDES BENZ', 'MERCEDES-BENZ'
*   'MITSUBISHI', 'MITSUBISHI MOTORS THAILAND'


In [ ]:
replace_mk = {'MERCEDES-BENZ' : 'MERCEDES BENZ',
              'MITSUBISHI MOTORS THAILAND' : 'MITSUBISHI'}
df['Marque'] = df['Marque'].replace(replace_mk)

In [ ]:
sorted(df['Marque'].unique())

### Analyse

In [ ]:
def analyser_variable_categorielle_plotly(df, variable, top_n=100, display_array=True):
  """
  Analyse une variable catégorielle en affichant un DataFrame des 'top_n' catégories les plus fréquentes, ainsi qu'un graphique.

  :param df: DataFrame contenant la variable à analyser.
  :param variable: Nom de la variable catégorielle.
  :param top_n: Nombre de catégories à afficher (par défaut 100).
  """

  if display_array == True:
    top_cat = f"(Top {top_n} catégories)" if top_n != 100 else ""
    print(f"\n Analyse de la variable : {variable} {top_cat}")

  # Calcul des fréquences et pourcentages
  category_counts = df[variable].value_counts().head(top_n)
  category_percent = df[variable].value_counts(normalize=True).head(top_n) * 100

  # Création d’un DataFrame avec Libellé, Total et Pourcentage
  df_summary = pd.DataFrame({
      "Libellé": category_counts.index,
      "Total": category_counts.values,
      "Pourcentage": category_percent.values
  })

  # Affichage du tableau de synthèse
  display(df_summary)

  # Création du graphique interactif avec Plotly
  nb = top_n
  if top_n == 100:
    nb = ""
  fig = px.bar(df_summary,
                x="Libellé",
                y="Total",
                text="Total",
                title=f'Distribution des {nb} premières catégories de {variable}',
                labels={"Libellé": variable, "Total": "Nombre d'occurrences"},
                template="plotly_white")

  fig.update_traces(textposition='outside')
  fig.update_layout(xaxis_tickangle=-45)

  # Affichage du graphique
  fig.show()


In [ ]:
analyser_variable_categorielle_plotly(df, 'Marque', 50)

## Analyse par carburant

### Valeurs uniques

In [ ]:
sorted(df['Carburant'].unique())

### Remplacement

Certaines valeurs peuvent être regroupées :




*   'lpg' & 'ng' sont des énergies alternatives


In [ ]:
replace_ft = {'lpg' : 'gaz',
              'ng' : 'gaz'}
df['Carburant'] = df['Carburant'].replace(replace_ft)

In [ ]:
sorted(df['Carburant'].unique())

### Analyse

In [ ]:
analyser_variable_categorielle_plotly(df, 'Carburant')

## Analyse par modèle de voiture

In [ ]:
analyser_variable_categorielle_plotly(df, 'Modèle', 30)

# <font color='#3585CD'>Distribution des variables numériques</font>

## Sélection des colonnes numériques

In [ ]:
# Sélection des colonnes numériques
num_vars = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_vars

## Analyse de la masse du véhicule Masse à vide & Masse totale

In [ ]:
from scipy.stats import gaussian_kde
def analyser_variables_numeriques_plotly(df, variables, bins=30):
    """
    Analyse les variables numériques en affichant un histogramme + KDE (distribution) et un boxplot interactifs.

    :param df: DataFrame contenant les données.
    :param variables: Liste des variables numériques à analyser.
    :param bins: Nombre de bins pour l'histogramme (par défaut 30).
    """
    for var in variables:
        #print(f"\n Analyse de la variable : {var}")
        #display(df[var].describe())  # Affichage des statistiques descriptives

        # Supprimer les valeurs NaN
        data = df[var].dropna()

        # Histogramme
        hist = go.Histogram(
            x=data,
            nbinsx=bins,
            marker=dict(color='skyblue', line=dict(color='black', width=1)),  # Bordures noires
            opacity=0.6,  # Semi-transparent pour voir la KDE
            name="Histogramme"
        )

        # Calcul des densités pour la courbe KDE
        kde = gaussian_kde(data)
        x_vals = np.linspace(data.min(), data.max(), 500)  # Intervalle lissé
        kde_vals = kde(x_vals)

        # Courbe KDE
        kde_curve = go.Scatter(
            x=x_vals,
            y=kde_vals * len(data) * (data.max() - data.min()) / bins,  # Mise à l'échelle par rapport à l'histogramme
            mode='lines',
            line=dict(color='blue', width=2),
            name="Densité (KDE)"
        )

        # Création de la figure combinée
        fig = go.Figure(data=[hist, kde_curve])

        # Mise en page
        fig.update_layout(
            title=f'Distribution de {var} (Histogramme + KDE)',
            xaxis_title="Valeur",
            yaxis_title="Fréquence",
            template="plotly_white",
            barmode='overlay'
        )

        # Affichage du graphique combiné
        fig.show()

        # Création du boxplot
        boxplot = go.Box(
            x=data,
            marker=dict(color='salmon'),
            name="Boxplot",
            boxpoints="outliers"  # Affichage des outliers
        )

        # Création et affichage du Boxplot
        fig_box = go.Figure(data=[boxplot])
        fig_box.update_layout(
            title=f'Boxplot de {var}',
            xaxis_title="Valeur",
            template="plotly_white"
        )

        fig_box.show()

In [ ]:
analyser_variables_numeriques_plotly(df, ['Masse à vide'])

In [ ]:
analyser_variables_numeriques_plotly(df, ['Masse totale'])

## Analyse des émissions spécifiques de CO2

In [ ]:
analyser_variables_numeriques_plotly(df, ['CO2'], 30)

## Analyse de la cylindrée moteur

In [ ]:
analyser_variables_numeriques_plotly(df, ['Cylindrée moteur'], 50)

## Analyse de la puissance du moteur

In [ ]:
analyser_variables_numeriques_plotly(df, ['Puissance moteur'])

## Analyse de la Consommation carburant

In [ ]:
analyser_variables_numeriques_plotly(df, ['Consommation carburant'])

# <font color='#3585CD'>Corrélation entre Masse à vide et Masse totale</font>

In [ ]:
def plot_correlation_matrix(df):
  """
  Affiche la matrice de corrélation des variables numériques sous forme de heatmap interactive avec Plotly.

  :param df: DataFrame Pandas contenant les données
  """
  # Sélection des colonnes numériques
  num_numeric_cols = df.select_dtypes(include=['number']).columns

  # Calcul de la matrice de corrélation
  corr_matrix = df[num_numeric_cols].corr()

  # Création de la heatmap avec Plotly (labels en bas et à gauche)
  fig = ff.create_annotated_heatmap(
      z=corr_matrix.values,
      x=list(corr_matrix.columns),
      y=list(corr_matrix.index),
      colorscale="RdBu_r",
      annotation_text=corr_matrix.round(2).values,
      showscale=True
  )

  # Ajustement de la disposition
  fig.update_layout(
      title="Matrice de corrélation des variables numériques",
      height=600, width=800,
      xaxis=dict(side="bottom", tickangle=-45),
      yaxis=dict(side="left")
  )

  # Affichage
  fig.show()

In [ ]:
df_masse = df[['Masse à vide', 'Masse totale']]
plot_correlation_matrix(df_masse)

On se rend compte qu'il y a une **forte corrélataion** entre ces 2 variables, qui pourrait entrainer une **colinéarité**. Nous pouvons faire la moyenne des masses puis supprimer les 2 variables Masse à vide et Masse totale

In [ ]:
df['Masse moyenne'] = (df['Masse à vide'] + df['Masse totale']) / 2

In [ ]:
df = df.drop(columns=['Masse à vide', 'Masse totale'], axis=1)

In [ ]:
plot_correlation_matrix(df)

# <font color='#3585CD'>Analyse du CO2 en fonction de certaines variables</font>

### Analyse du CO2 en fonction de la masse du véhicule

In [ ]:
def plot_scatter_co2(df, x, y="CO2", color="Carburant", size="CO2"):
  """
  Génère un scatter plot interactif avec Plotly, avec une ligne de moyenne CO2.

  Paramètres :
  - df : DataFrame contenant les données
  - x : Nom de la colonne pour l'axe X
  - y : Nom de la colonne pour l'axe Y (par défaut "CO2")
  - color : Nom de la colonne pour la couleur des points (par défaut "Carburant")
  - size : Nom de la colonne pour la taille des points (par défaut "CO2")
  """

  # Calcul de la moyenne globale du CO2
  moyenne_co2 = df[y].mean()

  # Création du scatter plot
  fig = px.scatter(
      df,
      x=x,
      y=y,
      color=color,
      size=size,
      title=f"Relation entre {x} et {y}",
      labels={x: x.capitalize(), y: y.capitalize(), color: color.capitalize()},
      hover_data=df.columns,
      size_max=20
  )

  # Ajout de la ligne de moyenne CO2
  fig.add_hline(
      y=moyenne_co2,
      line_dash="dot",
      line_color="red",
      annotation_text=f"Moyenne CO2: {moyenne_co2:.2f} g/km",
      annotation_position="top right",
      annotation_font_color="red",
      annotation_font_size=12,
      annotation_bgcolor="rgba(255,255,255,0.7)"
  )

  fig.show()


In [ ]:
plot_scatter_co2(df, "Masse moyenne")

On observe que les véhicules plus lourds tendent à émettre plus de CO₂, avec une distinction entre les types de carburant.

### Analyse du CO2 en fonction de la puissance du moteur.

In [ ]:
plot_scatter_co2(df, "Puissance moteur")

### Analyse du CO2 en fonction du carburant



In [ ]:
plot_scatter_co2(df, "Carburant")

In [ ]:
def analyser_hist_co2_par_variable(df, variable, top_n=50, order='desc', co2="CO2"):
  """
  Génère un histogramme interactif avec Plotly, avec une ligne de moyenne CO2.

  Paramètres :
  :param df : DataFrame contenant les données
  :param variable : Nom de la colonne à analyser
  :param top_n: Nombre de catégories à afficher (par défaut 50).
  :param order: Ordonnancement des catégories (par défaut 'desc').
  :param co2: Nom de la colonne des émissions de CO2 (par défaut "CO2")
  """
  ascending = True if order == 'asc' else False

  df_co2 = df.groupby(variable)[co2].mean().sort_values(ascending=ascending).reset_index()
  df_co2 = df_co2.head(top_n)

  df_co2['CO2_txt'] = df_co2[co2].apply(lambda x: f"{x:.2f}")

  moyenne_co2 = df[co2].mean()

  fig = px.bar(
      df_co2,
      x=variable,
      y=co2,
      title=f"Distribution des émissions de CO2 par {variable} (Top {top_n})",
      labels={variable: variable.capitalize(), co2: "Émissions de CO2 (g/km)"},
      color=co2,
      color_continuous_scale="RdYlGn_r",
      text='CO2_txt'
  )

  # Ligne de moyenne du CO2
  fig.add_hline(
      y=moyenne_co2,
      line_dash="dot",
      line_color="red",
      annotation_text=f"Moyenne CO2: {moyenne_co2:.2f} g/km",
      annotation_position="top right",
      annotation_font_color="red",
      annotation_font_size=12,
      annotation_bgcolor="rgba(255,255,255,0.7)"
  )

  fig.update_layout(
      xaxis_tickangle=-75,
      coloraxis_colorbar=dict(title="CO2 (g/km)"),
      uniformtext_minsize=8,
      uniformtext_mode='hide'
  )

  fig.show()

In [ ]:
analyser_hist_co2_par_variable(df, 'Carburant')

### Analyse du CO2 en fonction de la marque

In [ ]:
analyser_hist_co2_par_variable(df, 'Marque', 50)

### Analyse du CO2 en fonction du modèle de voiture

In [ ]:
analyser_hist_co2_par_variable(df, 'Modèle', 50)

### Analyse du CO2 en fonction de la consommation

In [ ]:
plot_scatter_co2(df, "Consommation carburant")

# <font color='#3585CD'>Corrélation des variables numériques & Suppression des variables non pertinentes</font>

In [ ]:
plot_correlation_matrix(df)

In [ ]:
def calculer_correlation(df, col1, col2):
  """
  Calcule et affiche la corrélation entre deux colonnes d'un DataFrame.

  Paramètres :
  df :DataFrame contenant les données.
  col1 : nom de la première colonne.
  col2 : nom de la deuxième colonne.

  Retourne la valeur de la corrélation et une interprétation de son intensité.
  """
  correlation = df[col1].corr(df[col2])
  print(f"Corrélation entre {col1} et {col2} : {correlation:.2f}")

  if abs(correlation) > 0.8:
      interpretation = "Très forte corrélation"
  elif abs(correlation) > 0.6:
      interpretation = "Forte corrélation"
  elif abs(correlation) > 0.4:
      interpretation = "Corrélation modérée"
  elif abs(correlation) > 0.2:
      interpretation = "Corrélation faible"
  else:
      interpretation = "Pas de corrélation linéaire significative"

  print(interpretation + ".\n")

  return correlation, interpretation

In [ ]:
num_numeric_cols = df.select_dtypes(include=['number']).columns
num_numeric_cols

for col in num_numeric_cols:
  calculer_correlation(df, col, 'CO2')

Ou RFE ?
Ou SelectKBest après la séparation des données ?
SelectFromModel

In [ ]:
# df = df.drop('Puissance moteur', axis=1)

In [ ]:
# df.duplicated().sum()

In [ ]:
# df = df.drop_duplicates()

In [ ]:
# plot_correlation_matrix(df)

# <font color='red'>Corrélation entre Cylindrée moteur et Puissance moteur</font>

In [ ]:
df_moteur = df[['Cylindrée moteur', 'Puissance moteur']]
plot_correlation_matrix(df_moteur)

# <font color='#3585CD'>Analyse des outliers</font>

In [ ]:
def detecter_outliers_plotly(df, seuil=1.5):
  """
  Détecte les outliers pour chaque variable numérique d'un DataFrame en utilisant la méthode IQR.
  Affiche également un Boxplot interactif pour chaque variable.

  :param df: DataFrame contenant les données.
  :param seuil: Seuil du coefficient IQR (par défaut 1.5).
  :return: DataFrame contenant le nombre d'outliers et le pourcentage par variable.
  """
  outliers_dict = {}

  for col in df.select_dtypes(include=['number']).columns:  # Sélectionner les colonnes numériques
      Q1 = df[col].quantile(0.25)  # Premier quartile
      Q3 = df[col].quantile(0.75)  # Troisième quartile
      IQR = Q3 - Q1  # Calcul de l'intervalle interquartile

      # Détection des valeurs aberrantes
      lower_bound = Q1 - seuil * IQR
      upper_bound = Q3 + seuil * IQR
      outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

      # Stocker les résultats
      nb_outliers = outliers.shape[0]
      perc_outliers = (nb_outliers / df.shape[0]) * 100
      outliers_dict[col] = {"Nb_Outliers": nb_outliers, "Pourcentage": round(perc_outliers, 2)}

      # Création du Boxplot avec Plotly
      fig = go.Figure()
      fig.add_trace(go.Box(
          x=df[col],
          name=col,
          marker_color='blue',
          boxpoints='outliers'
      ))

      # Mise en page
      fig.update_layout(
          title=f"Box Plot de {col}",
          xaxis_title="Valeur",
          yaxis_title="Variable",
          template="plotly_white",
          showlegend=False
      )

      fig.show()

  # Conversion en DataFrame
  df_outliers = pd.DataFrame.from_dict(outliers_dict, orient='index')

  return df_outliers

In [ ]:
detecter_outliers_plotly(df)

In [ ]:
df_outliers_masse = df[df['Masse moyenne'] > 2500].sort_values(by='Masse moyenne', ascending=True)
df_outliers_masse.tail(20)

In [ ]:
df_outliers_CO2 = df[df['CO2'] > 350].sort_values(by='CO2', ascending=True)
df_outliers_CO2.tail(20)

In [ ]:
df_outliers_Consommation = df[df['Consommation carburant'] > 15].sort_values(by='Consommation carburant', ascending=True)
df_outliers_Consommation.tail(20)

# <font color='#3585CD'>Visualisation globale graphique</font>

In [ ]:
fig = px.scatter_matrix(df, dimensions=df.select_dtypes(include=['number']).columns,
                        color='Carburant', title="Pairplot")

fig.update_layout(height=900, width=1200)
fig.show()

In [ ]:
plot_correlation_matrix(df)

In [ ]:
df = df.reset_index(drop=True)
df

# <font color='#3585CD'>Distribution de la variable cible</font>





## Histogramme et boxplot de la variable cible

In [ ]:
analyser_variables_numeriques_plotly(df, ['CO2'])

In [ ]:
# from scipy.stats import boxcox

# df['co2_transformed'], lambda_boxcox = boxcox(df['Ewltp (g/km)'] + 1)  # Ajouter 1 pour éviter 0
# df

In [ ]:
df['Carburant'].unique()

In [ ]:
# Histogramme en fonction du type de motorisation
fig = px.histogram(df, x="CO2", color="Carburant", nbins=50, barmode="overlay",
                   title="Distribution des émissions de CO2 par type de carburant")
fig.show()


Comment gérer ?


*   Création colonne Hybride ?
*   Dataset à part ?
*   Juste un OneHotEncoder ?

In [ ]:
# df["Hybride"] = (df["Ft"] == "hybride").astype(int)
# df

In [ ]:
# plot_scatter_co2(df, "Hybride")

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# from scipy.stats import skew, kurtosis

# # Histogramme avec courbe de densité
# plt.figure(figsize=(8, 5))
# sns.histplot(df['CO2'], bins=30, kde=True)
# plt.title("Distribution de co2_transformed")
# plt.show()

# # Boxplot pour détecter les valeurs extrêmes
# plt.figure(figsize=(8, 3))
# sns.boxplot(x=df['CO2'])
# plt.title("Boxplot de CO2")
# plt.show()

# # Calcul des statistiques
# skewness = skew(df['CO2'])
# kurt = kurtosis(df['CO2'])

# # Affichage des valeurs
# print(f"Asymétrie (skewness) : {skewness:.2f}")
# print(f"Aplatissement (kurtosis) : {kurt:.2f}")

# # Interprétation de l'asymétrie (skewness)
# if abs(skewness) < 0.2:
#     interpretation_skew = "très faible"
# elif abs(skewness) < 0.5:
#     interpretation_skew = "faible"
# elif abs(skewness) < 1:
#     interpretation_skew = "modérée"
# elif abs(skewness) < 2:
#     interpretation_skew = "forte"
# else:
#     interpretation_skew = "très forte"

# if skewness > 0:
#     print(f"L'asymétrie est **{interpretation_skew}** et **positive** (longue traîne à droite).")
# elif skewness < 0:
#     print(f"L'asymétrie est **{interpretation_skew}** et **négative** (longue traîne à gauche).")
# else:
#     print("La distribution est parfaitement symétrique.")

# # Interprétation de l'aplatissement (kurtosis)
# if kurt < 0:
#     interpretation_kurt = "extrêmement platykurtique (distribution très aplatie)"
# elif kurt < 1:
#     interpretation_kurt = "platykurtique (moins de pics, distribution plus large)"
# elif kurt < 3:
#     interpretation_kurt = "mésokurtique (distribution normale)"
# elif kurt < 10:
#     interpretation_kurt = "légerement leptokurtique (pics modérés, plus concentré)"
# else:
#     interpretation_kurt = "extrêmement leptokurtique (forte concentration autour de la moyenne, pics très élevés)"

# print(f"La distribution est **{interpretation_kurt}**.")


## Degrés d'asymétrie des variables

In [ ]:
num_vars = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
df[num_vars].skew().sort_values()

### CO2

Légère asymétrie négative (peu inquiétante)

### Consommation carburant

 Quasi symétrique (très léger)

### Masse moyenne

Asymétrie positive modérée à forte, probablement due à des valeurs élevées qui tirent la distribution.

Modèles à tester :
LinearRegression
LinearRegression poly ?
SGDRegressor ou LinearSVR (SVM)
DecisionTreeRegressor
RandomForestRegressor
GradientBoostinggRegressor
XGBRegresssor